# FAERS PostgreSQL Exploration

This notebook runs the exploration sections defined in `data/faers_explore.py` against the PostgreSQL `faers` schema.

Prerequisites:
- `.env` contains `POSTGRES_URI`
- the FAERS tables have already been loaded into PostgreSQL
- the notebook kernel uses the MedLens project environment

In [1]:
import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'faers_explore.py').exists():
    raise RuntimeError('Open this notebook from the MedLens workspace root so imports resolve correctly.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/home/ashu/github/MedLens')

In [2]:
from data import faers_explore as explore

importlib.reload(explore)

print(f'Using schema: {explore.SCHEMA}')
explore.query("SELECT current_database() AS database_name, current_user AS db_user, current_schema() AS current_schema")

Using schema: faers


,database_name,db_user,current_schema
0,medlens,medadmin,public


## 1. Schema Overview
Row counts, table sizes, and the loaded event date range.

In [3]:
schema_counts = explore.section_schema_overview()
schema_counts


  1. Schema overview — tables in `faers`
table_name   size  est_rows
      drug 179 MB   1909327
      reac  91 MB   1445416
      indi  86 MB   1186115
      ther  36 MB    594449
      demo  33 MB    406184
      outc  15 MB    295044

  1b. Exact row counts
table    rows
 drug 1909327
 reac 1445416
 indi 1186115
 ther  594449
 demo  406184
 outc  295044

  1c. Date range of loaded reports
earliest_event latest_event  distinct_months
        102312       502309              344


,table,rows
0,drug,1909327
1,reac,1445416
2,indi,1186115
3,ther,594449
4,demo,406184
5,outc,295044


## 2. Null Rates
Coverage for the MedLens-relevant columns across each FAERS table.

In [4]:
null_rates = explore.section_null_rates()
null_rates


  2. Null rates for MedLens-relevant columns
table           column   total  populated  pct_populated
 demo              age  406184     248412           61.2
 demo          age_grp  406184     122139           30.1
 demo              sex  406184     343640           84.6
 demo               wt  406184      68664           16.9
 demo reporter_country  406184     406184          100.0
 demo         occp_cod  406184     399340           98.3
 drug         drugname 1909327    1909327          100.0
 drug          prod_ai 1909327    1875580           98.2
 drug         role_cod 1909327    1909327          100.0
 drug            route 1909327    1376209           72.1
 drug         dose_amt 1909327     794257           41.6
 drug        dose_unit 1909327     794257           41.6
 reac               pt 1445416    1445416          100.0
 reac     drug_rec_act 1445416       1082            0.1
 outc         outc_cod  295044     295044          100.0
 ther         start_dt  594449     551638 

,table,column,total,populated,pct_populated
0,demo,age,406184,248412,61.2
1,demo,age_grp,406184,122139,30.1
2,demo,sex,406184,343640,84.6
3,demo,wt,406184,68664,16.9
4,demo,reporter_country,406184,406184,100.0
5,demo,occp_cod,406184,399340,98.3
6,drug,drugname,1909327,1909327,100.0
7,drug,prod_ai,1909327,1875580,98.2
8,drug,role_cod,1909327,1909327,100.0
9,drug,route,1909327,1376209,72.1


## 3. Role Codes
How FDA labels each drug mention: primary suspect, secondary suspect, concomitant, or interacting.

In [5]:
role_codes = explore.section_role_codes()
role_codes


  3. role_cod distribution (which drugs FDA blames)
role_cod  mentions   pct
      SS    819690 42.93
       C    672903 35.24
      PS    406273 21.28
       I     10461  0.55

  3b. FDA-flagged drug-drug interaction cases (role='I')
 ddi_cases  ddi_drug_mentions
      2327              10461


,role_cod,mentions,pct
0,SS,819690,42.93
1,C,672903,35.24
2,PS,406273,21.28
3,I,10461,0.55


## 4. Outcome Severity
FDA outcome codes grouped into the Major, Moderate, and Minor buckets used in MedLens.

In [6]:
outcomes = explore.section_outcomes()
outcomes


  4. outc_cod distribution — training severity labels
severity outc_cod      n
   Major       HO  82534
   Major       DE  32303
   Major       LT  11990
Moderate       DS   6112
Moderate       RI    981
Moderate       CA    895
   Minor       OT 160229

  4b. Cases with multiple outcomes (e.g. DE + HO)
 n_outcomes  cases
          1 162690
          2  49210
          3   8624
          4   1280
          5    418
          6    142


,severity,outc_cod,n
0,Major,HO,82534
1,Major,DE,32303
2,Major,LT,11990
3,Moderate,DS,6112
4,Moderate,RI,981
5,Moderate,CA,895
6,Minor,OT,160229


## 5. Drug Name Normalization
Compare reporter-entered brand names with FDA-normalized active ingredients.

In [7]:
drug_names = explore.section_drug_names()
drug_names


  5. Drug name vs active ingredient
 total_mentions  distinct_drugnames  distinct_ingredients pct_ai_populated
        1909327               27674                  6431             98.2

  5b. Top 20 active ingredients by mention count
                 prod_ai  mentions  n_brand_variants
             TIRZEPATIDE     76176                 3
         INFLIXIMAB-DYYB     41639                 2
               DUPILUMAB     33387                 2
              PREDNISONE     22792                 4
           ACETAMINOPHEN     21185                31
            METHOTREXATE     20103                 3
             VEDOLIZUMAB     20060                 2
              OMALIZUMAB     16025                 2
               RITUXIMAB     14711                 3
              EVOLOCUMAB     14039                 2
             TOCILIZUMAB     13439                 3
            LENALIDOMIDE     13303                 2
             TEDUGLUTIDE     11669                 1
              ADALIMU

,prod_ai,mentions,n_brand_variants
0,TIRZEPATIDE,76176,3
1,INFLIXIMAB-DYYB,41639,2
2,DUPILUMAB,33387,2
3,PREDNISONE,22792,4
4,ACETAMINOPHEN,21185,31
5,METHOTREXATE,20103,3
6,VEDOLIZUMAB,20060,2
7,OMALIZUMAB,16025,2
8,RITUXIMAB,14711,3
9,EVOLOCUMAB,14039,2


## 6. Polypharmacy Signal
Distribution of suspect-drug counts per case and how they align with outcome severity.

In [8]:
polypharmacy = explore.section_polypharmacy()
polypharmacy


  6. Polypharmacy: distribution of suspect drugs per case
         bucket  cases  pct
1 (monotherapy) 331393 81.6
        2 drugs  35835  8.8
        3 drugs  14924  3.7
      4-5 drugs  12218  3.0
     6-10 drugs   7818  1.9
      11+ drugs   3996  1.0

  6b. Multi-drug cases × severity cross-tab
           severity  multi_drug_cases
       6 DE (death)             10443
 5 LT (life-threat)              4433
4 HO (hospitalized)             22332
  3 DS (disability)               639
2 CA (cong anomaly)               254
  1 RI (req interv)                30
       0 OT (other)             27536
         no outcome              9124


,bucket,cases,pct
0,1 (monotherapy),331393,81.6
1,2 drugs,35835,8.8
2,3 drugs,14924,3.7
3,4-5 drugs,12218,3.0
4,6-10 drugs,7818,1.9
5,11+ drugs,3996,1.0


## 7. Drug-Drug Interaction Pairs
Top co-mentioned ingredient pairs in FDA-flagged interaction cases and in all suspect cases.

In [9]:
ddi_pairs = explore.section_ddi_pairs()
ddi_pairs


  7. Top drug pairs in multi-drug suspect cases (FDA role='I')
                  drug_a                    drug_b  co_mentions
  NIRMATRELVIR\RITONAVIR                TACROLIMUS           31
               LORAZEPAM               RISPERIDONE           22
              PREDNISONE                 RITUXIMAB           21
            ARIPIPRAZOLE                 CLOZAPINE           18
            ARIPIPRAZOLE                OLANZAPINE           18
               CLOZAPINE                OLANZAPINE           18
           ACETAMINOPHEN      ESCITALOPRAM OXALATE           17
               LORAZEPAM       QUETIAPINE FUMARATE           17
    SACUBITRIL\VALSARTAN               TIRZEPATIDE           16
              QUETIAPINE               RISPERIDONE           15
            ARIPIPRAZOLE               RISPERIDONE           15
              OLANZAPINE               RISPERIDONE           14
FLUOXETINE HYDROCHLORIDE VENLAFAXINE HYDROCHLORIDE           14
     QUETIAPINE FUMARATE               R

,drug_a,drug_b,co_mentions
0,NIRMATRELVIR\RITONAVIR,TACROLIMUS,31
1,LORAZEPAM,RISPERIDONE,22
2,PREDNISONE,RITUXIMAB,21
3,ARIPIPRAZOLE,CLOZAPINE,18
4,ARIPIPRAZOLE,OLANZAPINE,18
5,CLOZAPINE,OLANZAPINE,18
6,ACETAMINOPHEN,ESCITALOPRAM OXALATE,17
7,LORAZEPAM,QUETIAPINE FUMARATE,17
8,SACUBITRIL\VALSARTAN,TIRZEPATIDE,16
9,QUETIAPINE,RISPERIDONE,15


## 8. Drug to Reaction Patterns
Common reported reactions for a few representative high-signal drugs.

In [10]:
explore.section_drug_reactions()


  8. Top reactions for selected drugs

-- ACETAMINOPHEN --
                            reaction    n
                       Off label use 1994
                    Drug ineffective 1438
Product use in unapproved indication 1191
                            Dyspnoea 1181
                                Pain 1169
                             Fatigue 1053
                              Nausea 1053
                            Vomiting 1048

-- ASPIRIN --
             reaction   n
             Dyspnoea 622
        Off label use 422
                Cough 407
                 Fall 346
                 Pain 332
               Nausea 312
     Drug ineffective 309
Drug hypersensitivity 308

-- WARFARIN --
                                         reaction   n
                                    Off label use 223
                             Condition aggravated 220
                                           Stress 182
             Product use in unapproved indication 162
                           

## 9. Demographics by Severity
Age-group and sex breakdowns for major, moderate, and minor outcomes.

In [11]:
demographics = explore.section_demographics()
demographics


  9. Age group × severity
age_grp  major  moderate  minor  total
unknown  84487      3657  81685 284045
      A  12332       532  13487  68128
      E  13292       263   9256  45157
      C    580        23    620   3941
      T    592        17    643   3692
      I    185        24    131    636
      N    240       119    199    585

  9b. Sex × severity
    sex  major  moderate  minor  total
      F  51599      2430  53717 204178
      M  46361      1525  35715 137899
unknown  13694       663  16544  62544
    UNK     54        17     45   1563


,age_grp,major,moderate,minor,total
0,unknown,84487,3657,81685,284045
1,A,12332,532,13487,68128
2,E,13292,263,9256,45157
3,C,580,23,620,3941
4,T,592,17,643,3692
5,I,185,24,131,636
6,N,240,119,199,585


## 10. Training Example Preview
Reconstruct a few multi-drug severe cases end-to-end and format them as candidate fine-tuning examples.

In [12]:
training_preview = explore.section_training_example_preview(limit=3)
training_preview


  10. Full case reconstruction — 3 sample training examples

────────────────────────────────────────────────────────────────────────
CASE 231897332
────────────────────────────────────────────────────────────────────────

[demographics]
 age age_cod age_grp  sex   wt wt_cod reporter_country occp_cod
None    None    None None None   None               DE       MD

[drugs]
 drug_seq role_cod       drugname        prod_ai        route dose_amt dose_unit
        1       PS   ATORVASTATIN   ATORVASTATIN         Oral      NaN       NaN
        2       SS   ROSUVASTATIN   ROSUVASTATIN         Oral      NaN       NaN
        3       SS       PRALUENT     ALIROCUMAB Subcutaneous       75        MG
        4       SS       PRALUENT     ALIROCUMAB          NaN      NaN       NaN
        5       SS BEMPEDOIC ACID BEMPEDOIC ACID         Oral      NaN       NaN
        6       SS        REPATHA     EVOLOCUMAB Subcutaneous      NaN       NaN
        7       SS        REPATHA     EVOLOCUMAB         

,primaryid
0,231897332
1,204831466
2,174967172


## 11. Data Quality Flags
Quick checks for missing active ingredients, orphan rows, and suspicious ingredient strings.

In [13]:
quality_flags = explore.section_quality_flags()
quality_flags


  11. Data quality red flags
 drug_null_ai  drug_null_name  drug_orphan_no_demo  reac_orphan_no_demo  unique_cases_demo  unique_cases_drug  unique_cases_outc
        33747               0                    0                    0             406184             406184             222364

  11b. Suspicious active-ingredient strings (potential OCR/data-entry noise)
                                                                                                                                 prod_ai    n
                                                                                                                POLYETHYLENE GLYCOL 3350 1838
                                                                                                   LUTETIUM LU-177 VIPIVOTIDE TETRAXETAN 1368
CYANOCOBALAMIN\DEXPANTHENOL\NIACINAMIDE\PYRIDOXINE HYDROCHLORIDE\RIBOFLAVIN 5^-PHOSPHATE SODIUM\THIAMINE HYDROCHLORIDE\VITAMIN B COMPLEX  911
                                                          POLYETHY

,drug_null_ai,drug_null_name,drug_orphan_no_demo,reac_orphan_no_demo,unique_cases_demo,unique_cases_drug,unique_cases_outc
0,33747,0,0,0,406184,406184,222364


## Ad Hoc Query Cell
Edit the SQL below when you want to inspect a table directly without changing the underlying script.

In [14]:
sql = f"""
SELECT primaryid, role_cod, drugname, prod_ai
FROM {explore.SCHEMA}.drug
WHERE prod_ai IS NOT NULL
LIMIT 20
"""

explore.query(sql)

,primaryid,role_cod,drugname,prod_ai
0,1001678125,PS,SANDOSTATIN,OCTREOTIDE ACETATE
1,1001678125,SS,SANDOSTATIN,OCTREOTIDE ACETATE
2,1001678125,SS,SANDOSTATIN,OCTREOTIDE ACETATE
3,1001678125,SS,SANDOSTATIN LAR DEPOT,OCTREOTIDE ACETATE
4,1001678125,SS,SANDOSTATIN LAR DEPOT,OCTREOTIDE ACETATE
5,1001678125,SS,SANDOSTATIN LAR DEPOT,OCTREOTIDE ACETATE
6,1001678125,SS,SANDOSTATIN LAR DEPOT,OCTREOTIDE ACETATE
7,1001678125,SS,SANDOSTATIN LAR DEPOT,OCTREOTIDE ACETATE
8,1001678125,SS,SANDOSTATIN LAR DEPOT,OCTREOTIDE ACETATE
9,1001678125,C,SANDOSTATIN,OCTREOTIDE ACETATE
